In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
data = np.load("go_chunks/chunk_0000.npz")
data

NpzFile 'go_chunks/chunk_0000.npz' with keys: positions, next_to_go, winner

In [2]:
def load_data(path):
    with np.load(path) as data:
        data_positions = data["positions"]
        data_labels = data["winner"]

        

    data_positions = np.transpose(data_positions, (0, 2, 3, 1)) 
    return data_positions, data_labels


train_positions, train_labels = load_data("go_chunks/chunk_0000.npz")
test_positions, test_labels = load_data("go_chunks/chunk_0001.npz")


In [ ]:
import os 

cuttoff=200

def load_data_tf(paths):
    for path in paths:
        with np.load(path) as data:
            data_positions = data["positions"]
            data_labels = data["winner"]

        data_positions = np.transpose(data_positions, (0, 2, 3, 1)) 
        yield data_positions, data_labels

paths = [f"go_chunks/{f}" for f in os.listdir("go_chunks") if f.endswith(".npz")][:cuttoff]
#paths= ["go_chunks/chunk_0000.npz","go_chunks/chunk_0002.npz","go_chunks/chunk_0003.npz"]
ds = tf.data.Dataset.from_generator(load_data_tf, args=(paths,),output_signature=(
        tf.TensorSpec(shape=(None, 19, 19, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.int64)
    ))

ds = ds.flat_map(lambda x, y: tf.data.Dataset.from_tensor_slices((x, y)))
#ds_batch = ds.unbatch().batch(256)

#train_positions, train_labels = next(iter(ds))

In [6]:
from tensorflow.keras import layers, models

model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(19, 19, 3)))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))


model.add(layers.Flatten())
model.add(layers.Dense(1, activation='sigmoid'))

model.summary()
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])


ds = ds.shuffle(10000).batch(256).prefetch(tf.data.AUTOTUNE)

model.fit(ds, epochs=2,validation_data=test_dataset)


C:\Users\puket\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 19, 19, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 9, 9, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 9, 9, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,345 (224.00 KB)

 Trainable params: 57,345 (224.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2


C:\Users\puket\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\backend\tensorflow\nn.py:1286: UserWarning: "`binary_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Sigmoid activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


  21947/Unknown 442s 20ms/step - accuracy: 0.8263 - loss: 0.3366

KeyboardInterrupt: 

In [10]:
model.save_weights('./checkpoints/mini_CNN.weights.h5')

In [4]:
import tensorflow as tf

train_dataset = tf.data.Dataset.from_tensor_slices((train_positions, train_labels))
test_dataset = tf.data.Dataset.from_tensor_slices((test_positions, test_labels))

BATCH_SIZE = 254
SHUFFLE_BUFFER_SIZE = 100

train_dataset = train_dataset.shuffle(SHUFFLE_BUFFER_SIZE).batch(BATCH_SIZE)
test_dataset = test_dataset.batch(BATCH_SIZE)

In [50]:
train_dataset

<_BatchDataset element_spec=(TensorSpec(shape=(None, 19, 19, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int8, name=None))>